In [ ]:
import onnxruntime as ort
import numpy as np


# Load the ONNX model
model_path = "model.onnx"
session = ort.InferenceSession(model_path)

# Prepare input data for inference
input_name = session.get_inputs()[0].name
input_data = np.random.rand(1, 1, 512, 512).astype(np.float32)

# Run inference
outputs = session.run(None, {input_name: input_data})

# Process the outputs
for o in outputs:
    print(o.shape)

In [ ]:
from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose
import rasterio as rio 
from pathlib import Path

import torch

import sys

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')



# Specify the path to model config and checkpoint file
config_file = 'config.py'
checkpoint_file = 'weights.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()


# Required by loader
def read_tif(file_path, band_indices):
    """
    Reads specified bands from a TIFF file.

    Parameters:
    - file_path (str): Path to the .tif file.
    - band_indices (list of int): Indices of the bands to read.

    Returns:
    - numpy.ndarray: A numpy array containing the stacked band data.
    """
    data = []
    with rio.open(file_path) as src:
        for index in band_indices:
            data.append(src.read(index))
    stacked_data = np.stack(data, axis=0)
    return np.transpose(stacked_data, (1, 2, 0))

DATA_PATH_VEN = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
DATA_PATH_SEN = '/Data_large/marine/Datasets/VDS2Raw/imgs'

TIFF_VEN = list(Path(DATA_PATH_VEN).rglob('*.tif'))
TIFF_SEN = list(Path(DATA_PATH_SEN).rglob('*.tif'))

Idx = 3
tiffSel = None
test_pipeline = Compose(test_pipeline)
if tiffSel is None:
    data_ = dict(img_path=TIFF_SEN[Idx], img_id=0)
    ORIGINAL_IMG = read_tif(file_path=TIFF_SEN[Idx], band_indices=[4])

else:
    data_ = dict(img_path=tiffSel, img_id=0)
    ORIGINAL_IMG = read_tif(file_path=tiffSel, band_indices=[4])
    

data_ = test_pipeline(data_)
data_['inputs'] = [data_['inputs']]
data_['data_samples'] = [data_['data_samples']]


# forward the model
with torch.no_grad():
    results = model.test_step(data_)[0]
    
results


# ONNX

In [20]:
Idx = 157
tiffSel = None
test_pipeline = Compose(test_pipeline)
if tiffSel is None:
    data_ = dict(img_path=TIFF_SEN[Idx], img_id=0)
    ORIGINAL_IMG = read_tif(file_path=TIFF_SEN[Idx], band_indices=[3])

else:
    data_ = dict(img_path=tiffSel, img_id=0)
    ORIGINAL_IMG = read_tif(file_path=tiffSel, band_indices=[3])
    
data_ = test_pipeline(data_)
data_['inputs'] = [data_['inputs']]
data_['data_samples'] = [data_['data_samples']]

input_data = data_['inputs'][0].unsqueeze(0)


In [23]:
input_data = input_data.cpu().numpy()

In [25]:
import torch.nn.functional as F

# Convert input_data back to a tensor
input_tensor = torch.from_numpy(input_data)

# Upsample the input_data to 2304x2304
upsampled_input = F.interpolate(input_tensor, size=(2304, 2304), mode='bilinear', align_corners=False)

# Convert back to numpy array if needed
upsampled_input_data = upsampled_input.cpu().numpy()

In [ ]:
import onnxruntime as ort
import numpy as np


# Load the ONNX model
model_path = "./export/end2end.onnx"
session = ort.InferenceSession(model_path)

# Prepare input data for inference
input_name = session.get_inputs()[0].name
# input_data = np.random.rand(1, 1, 2304, 2304).astype(np.float32)

# Run inference
outputs = session.run(None, {input_name: upsampled_input_data})
outputs
# Process the outputs
# for o in outputs:
#     print(o.shape)

In [ ]:
import onnxruntime as ort
import numpy as np


# Load the ONNX model
model_path = "model_apisonnx.onnx"
session = ort.InferenceSession(model_path)

# Prepare input data for inference
input_name = session.get_inputs()[0].name
input_data = np.random.rand(1, 1, 512, 512).astype(np.float32)

# Run inference
outputs = session.run(None, {input_name: input_data})

# Process the outputs
for o in outputs:
    print(o.shape)